# hrono — research notebook

Step 1: read `hrono.toml`, clone/update configured git sources into a local cache.

Cache layout mirrors the source URI scheme from the design sketch: `<source-type>://<instance>/<path...>` becomes `.hrono/cache/<source-type>/<instance>/<path...>` on disk — not a hardcoded `repos/` folder, so future source types (Contentful, Confluence) slot into the same cache mechanism.

In [1]:
import os
import subprocess
import tomllib
from pathlib import Path
from urllib.parse import urlparse

APP_DIR = Path.cwd()
CACHE_ROOT = APP_DIR / ".hrono" / "cache"
CONFIG_PATH = APP_DIR / "hrono.toml"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)


def parse_git_url(url: str) -> tuple[str, str, str]:
    """Return (host, org, repo) for either SSH (git@host:org/repo.git) or HTTPS URL forms."""
    if url.startswith("git@"):
        host, path_part = url[len("git@"):].split(":", 1)
    else:
        parsed = urlparse(url)
        host = parsed.netloc
        path_part = parsed.path.lstrip("/")
    if path_part.endswith(".git"):
        path_part = path_part[: -len(".git")]
    org, repo = path_part.split("/", 1)
    return host, org, repo


def cache_path_for_git(host: str, org: str, repo: str) -> Path:
    return CACHE_ROOT / "git" / host / org / repo


def resolve_clone_url(url: str, auth: str) -> str:
    """auth is 'none', 'ssh', or 'env:VAR_NAME' (token injected into an HTTPS URL)."""
    if auth.startswith("env:"):
        var_name = auth.split(":", 1)[1]
        token = os.environ.get(var_name)
        if not token:
            raise RuntimeError(f"env var {var_name} not set for auth on {url}")
        parsed = urlparse(url)
        return parsed._replace(netloc=f"{token}@{parsed.netloc}").geturl()
    return url


def clone_or_update(url: str, auth: str = "none") -> Path:
    host, org, repo = parse_git_url(url)
    dest = cache_path_for_git(host, org, repo)
    clone_url = resolve_clone_url(url, auth)
    if dest.exists():
        print(f"updating {host}/{org}/{repo}")
        subprocess.run(["git", "-C", str(dest), "fetch", "--all", "--prune"], check=True, capture_output=True, text=True)
    else:
        print(f"cloning {host}/{org}/{repo}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", clone_url, str(dest)], check=True, capture_output=True, text=True)
    return dest


# --- run: read config, clone/update every configured git source ---
with open(CONFIG_PATH, "rb") as f:
    config = tomllib.load(f)

results = []
for src in config.get("sources", {}).get("git", []):
    path = clone_or_update(src["url"], src.get("auth", "none"))
    results.append((src["url"], path))

print()
for url, path in results:
    print(f"{url} -> {path}")

updating github.com/octocat/Hello-World


updating github.com/octocat/Spoon-Knife


updating github.com/branow/webphone


updating github.com/branow/football-match-mvc



https://github.com/octocat/Hello-World.git -> /Users/orest/repos/rad/hrono/app/.hrono/cache/git/github.com/octocat/Hello-World
https://github.com/octocat/Spoon-Knife.git -> /Users/orest/repos/rad/hrono/app/.hrono/cache/git/github.com/octocat/Spoon-Knife
https://github.com/branow/webphone.git -> /Users/orest/repos/rad/hrono/app/.hrono/cache/git/github.com/branow/webphone
git@github.com:branow/football-match-mvc.git -> /Users/orest/repos/rad/hrono/app/.hrono/cache/git/github.com/branow/football-match-mvc


Step 2: filter files. `filter.toml` decides which files in a cloned source are even candidates for chunking — one glob-pattern list (`Path.full_match`, `**` across directories) covers vendored/generated dirs, known lockfiles, and binary extensions, plus an oversized-file cap and a null-byte sniff fallback for anything a pattern didn't catch.

In [2]:
FILTER_CONFIG_PATH = APP_DIR / "filter.toml"

with open(FILTER_CONFIG_PATH, "rb") as f:
    filter_config = tomllib.load(f)

_max_bytes = filter_config["limits"]["max_size_mb"] * 1024 * 1024
_excluded_patterns = filter_config["exclude"]["patterns"]
_sniff_enabled = filter_config["sniff"]["enabled"]

SNIFF_BYTES = 8000


def is_binary(path: Path) -> bool:
    with open(path, "rb") as f:
        head = f.read(SNIFF_BYTES)
    return b"\x00" in head


def should_chunk(path: Path, root: Path) -> bool:
    rel = path.relative_to(root)
    if any(rel.full_match(pattern) for pattern in _excluded_patterns):
        return False
    size = path.stat().st_size
    if size == 0 or size > _max_bytes:
        return False
    if _sniff_enabled and is_binary(path):
        return False
    return True


def walk_source(root: Path) -> list[Path]:
    return [p for p in root.rglob("*") if p.is_file() and should_chunk(p, root)]


# --- run: filter every cloned source, report keep/skip counts ---
filtered_files: dict[Path, list[Path]] = {}
for url, repo_path in results:
    kept = walk_source(repo_path)
    total = sum(1 for p in repo_path.rglob("*") if p.is_file())
    filtered_files[repo_path] = kept
    print(f"{repo_path.name}: {len(kept)}/{total} files kept")


Hello-World: 1/30 files kept
Spoon-Knife: 3/32 files kept
webphone: 387/423 files kept
football-match-mvc: 91/123 files kept


Step 3: chunk files. `chunking.toml` maps each file extension to a category (code/docs/data), each with its own target size and separator hierarchy — recursive delimiter splitting, LangChain `RecursiveCharacterTextSplitter` style, per design-sketch-v1.md. Each chunk gets a URI (`git://host/org/repo@commit/path#chunk-N`) and a SHA-256 content hash for later change detection.

In [3]:
import hashlib

CHUNKING_CONFIG_PATH = APP_DIR / "chunking.toml"

with open(CHUNKING_CONFIG_PATH, "rb") as f:
    chunking_config = tomllib.load(f)

_default_category = chunking_config["default"]["category"]
_ext_to_category = chunking_config["extensions"]
_categories = chunking_config["categories"]


def category_for(path: Path) -> str:
    return _ext_to_category.get(path.suffix.lower(), _default_category)


def split_text(text: str, separators: list[str], chunk_size: int, base_offset: int = 0) -> list[tuple[str, bool, int, int]]:
    """Recursively split on the first separator, falling back to the next
    separator (then raw chars) for any piece still over chunk_size. Returns
    (piece, is_raw, start, end): start/end are char offsets relative to the
    original file text, so a chunk's span can be recovered even though the
    reconstructed content (see merge_pieces) is a normalized rejoin, not a
    verbatim substring. is_raw marks pieces from the final char-level
    fallback — the only case with no real separator, so the only place a
    piece boundary can fall mid-word/mid-token."""
    if len(text) <= chunk_size or not separators:
        return [(text, False, base_offset, base_offset + len(text))] if text else []
    sep, *rest = separators
    if not sep:
        return [(ch, True, base_offset + i, base_offset + i + 1) for i, ch in enumerate(text)]
    result = []
    pos = base_offset
    parts = text.split(sep)
    for idx, piece in enumerate(parts):
        start, end = pos, pos + len(piece)
        if len(piece) > chunk_size:
            result.extend(split_text(piece, rest, chunk_size, start))
        elif piece:
            result.append((piece, False, start, end))
        pos = end + (len(sep) if idx < len(parts) - 1 else 0)
    return result


def merge_pieces(pieces: list[tuple[str, bool, int, int]], chunk_size: int) -> list[tuple[str, int, int]]:
    """Pack pieces up to chunk_size. Pieces from a real separator join with a
    newline; consecutive raw-char pieces (no real separator to land on) join
    with nothing, so a split token reassembles correctly instead of gaining a
    stray newline per character. No overlap: a cut only ever falls between
    two whole pieces, which are already clean except in the raw-char case
    this join rule already handles. Returns (content, char_start, char_end)
    per chunk — the span comes from the underlying pieces' original-file
    offsets, not from re-measuring the (possibly normalized) joined content."""
    chunks = []
    current, current_start, current_end, current_ends_raw = "", None, None, False
    for piece, is_raw, start, end in pieces:
        join = "" if (is_raw and current_ends_raw) else "\n"
        candidate = f"{current}{join}{piece}" if current else piece
        if len(candidate) <= chunk_size:
            current = candidate
            current_start = start if current_start is None else current_start
            current_end = end
        else:
            if current:
                chunks.append((current, current_start, current_end))
            current, current_start, current_end = piece, start, end
        current_ends_raw = is_raw
    if current:
        chunks.append((current, current_start, current_end))
    return chunks


def chunk_file(path: Path) -> list[dict]:
    category = category_for(path)
    cfg = _categories[category]
    chunk_size = cfg["chunk_size_kb"] * 1024
    raw_bytes = path.read_bytes()
    text = raw_bytes.decode("utf-8", errors="replace")
    pieces = split_text(text, cfg["separators"], chunk_size)
    chunks = []
    for content, char_start, char_end in merge_pieces(pieces, chunk_size):
        byte_start = len(text[:char_start].encode("utf-8"))
        byte_end = len(text[:char_end].encode("utf-8"))
        chunks.append({
            "content": content,
            "byte_start": byte_start,
            "byte_end": byte_end,
            "hash": hashlib.sha256(raw_bytes[byte_start:byte_end]).hexdigest(),
        })
    return chunks


def repo_head_hash(repo_path: Path) -> str:
    out = subprocess.run(
        ["git", "-C", str(repo_path), "rev-parse", "HEAD"],
        check=True, capture_output=True, text=True,
    )
    return out.stdout.strip()


def chunk_parts(repo_path: Path, file_path: Path, head_hash: str, index: int) -> dict:
    """The URI decomposed into its parts — the parts are the stored source of
    truth, the uri string is a render of them (a generated column in the DB,
    see step 7). host/org/repo come from the cache path, which parse_git_url
    already normalizes across SSH/HTTPS auth forms — so identity is keyed on
    the resource (repo+commit+path), never on how it was fetched. "git" is
    the resource's content model (commit-addressable file tree), not a
    transport — see design-sketch-v1.md. version is opaque and equality-only
    (git: commit sha; contentful: entry version; confluence: page version).
    chunk_no is a plain sequence number: its only job is ordering chunks to
    reassemble the file — cross-version chunk identity is the content hash,
    and @version already changes the whole URI on any edit, so a "stable"
    structural fragment name would buy nothing."""
    host, org, repo = repo_path.parts[-3], repo_path.parts[-2], repo_path.parts[-1]
    rel = file_path.relative_to(repo_path)
    return {
        "source_type": "git",
        "instance": host,
        "path": f"{org}/{repo}/{rel}",
        "version": head_hash,
        "chunk_no": index,
    }


def render_uri(parts: dict) -> str:
    """Canonical render, per design-sketch-v1.md's scheme:
    <source-type>://<instance>/<path>#chunk-<n>@<version>"""
    return (f"{parts['source_type']}://{parts['instance']}/{parts['path']}"
            f"#chunk-{parts['chunk_no']}@{parts['version']}")


# --- run: chunk every kept file, report counts + avg size per category ---
all_chunks = []
for repo_path, files in filtered_files.items():
    head_hash = repo_head_hash(repo_path)
    for file_path in files:
        for i, chunk in enumerate(chunk_file(file_path)):
            parts = chunk_parts(repo_path, file_path, head_hash, i)
            all_chunks.append({
                **parts,
                "uri": render_uri(parts),
                "content": chunk["content"],
                "byte_start": chunk["byte_start"],
                "byte_end": chunk["byte_end"],
                "hash": chunk["hash"],
                "category": category_for(file_path),
            })

print(f"{len(all_chunks)} chunks from {sum(len(f) for f in filtered_files.values())} files")
for cat in _categories:
    sizes = [len(c["content"]) for c in all_chunks if c["category"] == cat]
    if sizes:
        print(f"{cat}: {len(sizes)} chunks, avg {sum(sizes)//len(sizes)} chars, max {max(sizes)}")

510 chunks from 482 files
code: 419 chunks, avg 1990 chars, max 10211
docs: 61 chunks, avg 2798 chars, max 10087
data: 30 chunks, avg 1417 chars, max 2042


Step 4: which files actually got split into more than one chunk, and where. Groups the chunks back by source file (via the URI, stripping `#chunk-N`) and, for every file that needed more than one chunk, prints the actual boundary text — the tail of chunk N and the head of chunk N+1 — so a bad cut is visible directly instead of just inferred from counts.

In [4]:
from collections import Counter, defaultdict

chunks_by_file = defaultdict(list)
for c in all_chunks:
    # group by the file's identity parts directly — no URI string parsing
    file_key = f"{c['source_type']}://{c['instance']}/{c['path']}@{c['version']}"
    chunks_by_file[file_key].append(c)
for cs in chunks_by_file.values():
    cs.sort(key=lambda c: c["chunk_no"])

split_files = {f: cs for f, cs in chunks_by_file.items() if len(cs) > 1}

print(f"{len(chunks_by_file)} files chunked, {len(all_chunks)} chunks total")
print(f"{len(split_files)} files split into more than one chunk "
      f"({100 * len(split_files) / len(chunks_by_file):.1f}%)")

chunk_count_dist = Counter(len(cs) for cs in chunks_by_file.values())
for n in sorted(chunk_count_dist):
    print(f"  {n} chunk(s): {chunk_count_dist[n]} files")

BOUNDARY_WINDOW = 80
print()
print(f"=== split points (last {BOUNDARY_WINDOW} chars of chunk N | first {BOUNDARY_WINDOW} chars of chunk N+1) ===")
for file_key, cs in split_files.items():
    print(f"\n--- {file_key} ({len(cs)} chunks) ---")
    for i in range(len(cs) - 1):
        tail = cs[i]["content"][-BOUNDARY_WINDOW:]
        head = cs[i + 1]["content"][:BOUNDARY_WINDOW]
        print(f"  [end of chunk {i}]   ...{tail!r}")
        print(f"  [start of chunk {i + 1}] {head!r}...")

482 files chunked, 510 chunks total
17 files split into more than one chunk (3.5%)
  1 chunk(s): 465 files
  2 chunk(s): 10 files
  3 chunk(s): 3 files
  4 chunk(s): 4 files

=== split points (last 80 chars of chunk N | first 80 chars of chunk N+1) ===

--- git://github.com/branow/webphone/LICENSE@c6feb30927daddb0a18847819f083bf36ac30058 (4 chunks) ---
  [end of chunk 0]   ...' requirement modifies the requirement in section 4 to\n"keep intact all notices".'
  [start of chunk 1] 'c) You must license the entire work, as a whole, under this\nLicense to anyone wh'...
  [end of chunk 1]   ...'(including any patent licenses granted under the third\nparagraph of section 11).'
  [start of chunk 2] 'However, if you cease all violation of this License, then your\nlicense from a pa'...
  [end of chunk 2]   ...'a result of your choosing to follow a\nlater version.\n15. Disclaimer of Warranty.'
  [start of chunk 3] 'THERE IS NO WARRANTY FOR THE PROGRAM, TO THE EXTENT PERMITTED BY\nAPPLICABLE LAW.'

Step 5: export a chunk index — references only, no content. Each entry is `uri` (source + file path + chunk index, already unique per chunk), `byte_start`/`byte_end` (this chunk's span in the source file, for later exact re-extraction), and `hash` (SHA-256 of the raw file bytes in that span, not the reconstructed/normalized chunk text — so it reflects the real source exactly, for change detection against a future re-clone).

In [5]:
import json

CHUNK_INDEX_PATH = APP_DIR / ".hrono" / "chunk_index.json"

chunk_index = [
    {
        "uri": c["uri"],
        "byte_start": c["byte_start"],
        "byte_end": c["byte_end"],
        "hash": c["hash"],
    }
    for c in all_chunks
]

with open(CHUNK_INDEX_PATH, "w") as f:
    json.dump(chunk_index, f, indent=2)

print(f"wrote {len(chunk_index)} chunk references to {CHUNK_INDEX_PATH}")
print(json.dumps(chunk_index[:2], indent=2))


wrote 510 chunk references to /Users/orest/repos/rad/hrono/app/.hrono/chunk_index.json
[
  {
    "uri": "git://github.com/octocat/Hello-World/README#chunk-0@7fd1a60b01f91b314f59955a4e4d4e80d8edf11d",
    "byte_start": 0,
    "byte_end": 13,
    "hash": "03ba204e50d126e4674c005e04d82e84c21366780af1f43bd54a37816b6ab340"
  },
  {
    "uri": "git://github.com/octocat/Spoon-Knife/index.html#chunk-0@d0dd1f61b33d64e29d8bc1372a94ef6a2fee76a9",
    "byte_start": 0,
    "byte_end": 355,
    "hash": "2b3919976ed3c3d176da250161d0d1c71052d518c4e56d63bcc57343e0e754e9"
  }
]


Step 6: duplicate content detection. Two different `hrono.toml` sources (e.g. a GitHub URL and a local clone of the same repo, or just two repos that happen to vendor the same file) can produce chunks with different URIs but identical bytes. URI is provenance (where this reference came from) and stays per-source; dedup is a separate concern that groups by `hash` — same model as git's own content-addressed blobs, where paths point at objects and identical content across paths costs nothing extra. This doesn't change `chunk_index.json`'s shape (it already has `hash` per entry); it's the mechanism a future embedding step should use to embed each unique hash once, however many URIs point at it.

In [6]:
by_hash = defaultdict(list)
for c in chunk_index:
    by_hash[c["hash"]].append(c["uri"])

duplicate_groups = {h: uris for h, uris in by_hash.items() if len(uris) > 1}

print(f"{len(chunk_index)} chunk references, {len(by_hash)} unique content hashes")
print(f"{len(duplicate_groups)} hashes shared by more than one URI "
      f"({sum(len(u) for u in duplicate_groups.values())} references involved)")

for h, uris in duplicate_groups.items():
    print(f"\n--- hash {h[:12]}... ({len(uris)} references to the same content) ---")
    for uri in uris:
        print(f"  {uri}")


510 chunk references, 510 unique content hashes
0 hashes shared by more than one URI (0 references involved)


Step 7: persist chunks to SQLite — first table of the storage backend decided in `context.md` (SQLite + sqlite-vec; one file, one transaction across chunk metadata / tree edges / embeddings). The URI's decomposed parts (`source_type`, `instance`, `path`, `version`, `chunk_no`) are the stored columns — queries filter on parts (`WHERE path LIKE 'org/repo/%'`) instead of parsing a string — and `uri` is a STORED generated column, so the render can never drift from the parts. `version` is opaque and equality-only (commit sha / CMS entry version / wiki page version all serve identically); `chunk_no` only orders chunks for file reassembly — cross-version identity is `hash`, which is indexed as the dedup (step 6) and future embed-once key. References only, no content — content is re-extractable from the source cache via the byte range. Each run drops and rebuilds the table (schema is still evolving); per-chunk incremental upsert is the later update-mechanism work, not this step.

In [7]:
import sqlite3
import pandas as pd

DB_PATH = APP_DIR / ".hrono" / "hrono.db"

db = sqlite3.connect(DB_PATH)
db.executescript("""
DROP TABLE IF EXISTS chunks;
CREATE TABLE chunks (
    id          INTEGER PRIMARY KEY,
    source_type TEXT    NOT NULL,  -- content model: 'git' | 'contentful' | 'confluence' — never transport/auth
    instance    TEXT    NOT NULL,  -- 'github.com' | contentful space | atlassian site
    path        TEXT    NOT NULL,  -- slash-joined, depth varies per source; subtree queries via LIKE 'prefix/%'
    version     TEXT    NOT NULL,  -- opaque, equality-only: commit sha / entry version / page version
    chunk_no    INTEGER NOT NULL,  -- ordering for file reassembly; cross-version identity is hash
    byte_start  INTEGER NOT NULL,
    byte_end    INTEGER NOT NULL,
    hash        TEXT    NOT NULL,
    category    TEXT    NOT NULL,
    uri TEXT GENERATED ALWAYS AS (
        source_type || '://' || instance || '/' || path
        || '#chunk-' || chunk_no || '@' || version
    ) STORED,
    UNIQUE (source_type, instance, path, version, chunk_no)
);
CREATE INDEX idx_chunks_hash ON chunks(hash);
CREATE UNIQUE INDEX idx_chunks_uri ON chunks(uri);
""")

with db:  # one transaction: the rebuild is atomic, a crash can't leave a half-written table
    db.executemany(
        """INSERT INTO chunks (source_type, instance, path, version, chunk_no,
                               byte_start, byte_end, hash, category)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
        [(c["source_type"], c["instance"], c["path"], c["version"], c["chunk_no"],
          c["byte_start"], c["byte_end"], c["hash"], c["category"]) for c in all_chunks],
    )

count = db.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
print(f"{count} chunks stored in {DB_PATH}")
for row in db.execute(
    "SELECT category, COUNT(*), COUNT(DISTINCT hash) FROM chunks GROUP BY category ORDER BY category"
):
    print(f"  {row[0]}: {row[1]} chunks, {row[2]} unique hashes")

n = db.execute("SELECT COUNT(*) FROM chunks WHERE instance = 'github.com' AND path LIKE 'branow/webphone/client/%'").fetchone()[0]
print(f"subtree query, no string parsing — chunks under branow/webphone/client: {n}")

preview = pd.read_sql_query("SELECT * FROM chunks ORDER BY id LIMIT 8", db)
db.close()
preview

510 chunks stored in /Users/orest/repos/rad/hrono/app/.hrono/hrono.db
  code: 419 chunks, 419 unique hashes
  data: 30 chunks, 30 unique hashes
  docs: 61 chunks, 61 unique hashes
subtree query, no string parsing — chunks under branow/webphone/client: 234


,id,source_type,instance,path,version,chunk_no,byte_start,byte_end,hash,category,uri
0,1,git,github.com,octocat/Hello-World/README,7fd1a60b01f91b314f59955a4e4d4e80d8edf11d,0,0,13,03ba204e50d126e4674c005e04d82e84c21366780af1f4...,docs,git://github.com/octocat/Hello-World/README#ch...
1,2,git,github.com,octocat/Spoon-Knife/index.html,d0dd1f61b33d64e29d8bc1372a94ef6a2fee76a9,0,0,355,2b3919976ed3c3d176da250161d0d1c71052d518c4e56d...,docs,git://github.com/octocat/Spoon-Knife/index.htm...
2,3,git,github.com,octocat/Spoon-Knife/styles.css,d0dd1f61b33d64e29d8bc1372a94ef6a2fee76a9,0,0,256,027fb1facae93029fe75a530edcebfeccaa0be787b066e...,docs,git://github.com/octocat/Spoon-Knife/styles.cs...
3,4,git,github.com,octocat/Spoon-Knife/README.md,d0dd1f61b33d64e29d8bc1372a94ef6a2fee76a9,0,0,780,f93aba3ff091ee0b498ae165ff08453620f9802fc60986...,docs,git://github.com/octocat/Spoon-Knife/README.md...
4,5,git,github.com,branow/webphone/LICENSE,c6feb30927daddb0a18847819f083bf36ac30058,0,0,10128,2379c4755f188ca62f6614110a2b78808ffee29508c955...,docs,git://github.com/branow/webphone/LICENSE#chunk...
5,6,git,github.com,branow/webphone/LICENSE,c6feb30927daddb0a18847819f083bf36ac30058,1,10130,20151,35cc1d41131aad2b505fb5163b75a7b497af4c79af8489...,docs,git://github.com/branow/webphone/LICENSE#chunk...
6,7,git,github.com,branow/webphone/LICENSE,c6feb30927daddb0a18847819f083bf36ac30058,2,20153,30190,1dfdce255d900a7e417ce04757a735fcc09e3e0e585865...,docs,git://github.com/branow/webphone/LICENSE#chunk...
7,8,git,github.com,branow/webphone/LICENSE,c6feb30927daddb0a18847819f083bf36ac30058,3,30192,33893,8acd7ce587cd6a948e0183f926f56e747597a1287b61b8...,docs,git://github.com/branow/webphone/LICENSE#chunk...


Step 8: embeddings. `embedding.toml` names the model (any Hugging Face id — sentence-transformers downloads it on first use and exposes the same `encode()` for all of them, so the model is a config string, not code). Embedding is per unique `hash`, not per URI — the dedup/embed-once key from step 6. The `embeddings` table is keyed `(hash, model)`: content-addressed, so unlike `chunks` it is *not* dropped per run — re-running only embeds hashes not yet seen by the configured model (the first genuinely incremental piece of the pipeline), and a model swap in the TOML re-embeds everything for the new model instead of silently mixing vector spaces. Vectors are stored normalized (unit length), so similarity is a plain dot product. Ends with a sanity check: nearest neighbors of one sample chunk, which should be visibly on-topic.

In [8]:
import time

import numpy as np
from sentence_transformers import SentenceTransformer

EMBEDDING_CONFIG_PATH = APP_DIR / "embedding.toml"

with open(EMBEDDING_CONFIG_PATH, "rb") as f:
    embedding_config = tomllib.load(f)

_model_name = embedding_config["model"]["name"]
_batch_size = embedding_config["model"]["batch_size"]

db = sqlite3.connect(DB_PATH)
db.execute("""
CREATE TABLE IF NOT EXISTS embeddings (
    hash   TEXT    NOT NULL,  -- chunk content hash (chunks.hash) — content-addressed, survives re-runs
    model  TEXT    NOT NULL,  -- which model produced this vector; vectors from different models never mix
    dims   INTEGER NOT NULL,
    vector BLOB    NOT NULL,  -- float32 little-endian, dims * 4 bytes, unit-normalized
    PRIMARY KEY (hash, model)
)
""")

content_by_hash = {}
for c in all_chunks:
    content_by_hash.setdefault(c["hash"], c["content"])

done = {row[0] for row in db.execute("SELECT hash FROM embeddings WHERE model = ?", (_model_name,))}
todo = [h for h in content_by_hash if h not in done]
print(f"{len(content_by_hash)} unique hashes, {len(done)} already embedded by {_model_name}, {len(todo)} to embed")

t0 = time.monotonic()
model = SentenceTransformer(_model_name)
print(f"model load: {time.monotonic() - t0:.1f}s")


def embed(texts: list[str]) -> np.ndarray:
    return model.encode(texts, batch_size=_batch_size, normalize_embeddings=True, show_progress_bar=False)


if todo:
    start = time.monotonic()
    total_chars = 0
    # one transaction per batch, not one big one at the end — progress is watchable
    # live from another terminal: sqlite3 .hrono/hrono.db "SELECT COUNT(*) FROM embeddings;"
    for i in range(0, len(todo), _batch_size):
        batch = todo[i : i + _batch_size]
        vectors = embed([content_by_hash[h] for h in batch])
        total_chars += sum(len(content_by_hash[h]) for h in batch)
        with db:
            db.executemany(
                "INSERT INTO embeddings (hash, model, dims, vector) VALUES (?, ?, ?, ?)",
                [(h, _model_name, vectors.shape[1], np.asarray(v, dtype=np.float32).tobytes())
                 for h, v in zip(batch, vectors)],
            )
    elapsed = time.monotonic() - start
    print(f"embedded {len(todo)} chunks in {elapsed:.1f}s — "
          f"{1000 * elapsed / len(todo):.0f} ms/chunk, {len(todo) / elapsed:.1f} chunks/s, "
          f"{total_chars / elapsed / 1024:.0f} KB of text/s, {vectors.shape[1]} dims")

# --- benchmark: fresh system numbers every run, even when nothing is left to embed ---
sample = sorted(content_by_hash.values(), key=len, reverse=True)[:_batch_size]  # worst case: the biggest chunks
t0 = time.monotonic()
embed(sample)
bench = time.monotonic() - t0
print(f"benchmark (worst case, {len(sample)} biggest chunks, batch_size={_batch_size}): "
      f"{1000 * bench / len(sample):.0f} ms/chunk, {len(sample) / bench:.1f} chunks/s")

# --- sanity check: nearest neighbors of one sample chunk (vectors are unit length -> dot = cosine) ---
rows = db.execute("SELECT hash, vector FROM embeddings WHERE model = ?", (_model_name,)).fetchall()
hashes = [r[0] for r in rows]
matrix = np.frombuffer(b"".join(r[1] for r in rows), dtype=np.float32).reshape(len(rows), -1)
uri_by_hash = dict(db.execute("SELECT hash, uri FROM chunks").fetchall())

sample_hash = db.execute(
    "SELECT hash FROM chunks WHERE path LIKE '%webphone/README.md' AND chunk_no = 0"
).fetchone()[0]
sims = matrix @ matrix[hashes.index(sample_hash)]
print(f"\nnearest neighbors of {uri_by_hash[sample_hash]}:")
for j in np.argsort(-sims)[1:6]:
    print(f"  {sims[j]:.3f}  {uri_by_hash[hashes[j]]}")
db.close()

510 unique hashes, 510 already embedded by Qwen/Qwen3-Embedding-0.6B, 0 to embed


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

model load: 5.9s


benchmark (worst case, 8 biggest chunks, batch_size=8): 4814 ms/chunk, 0.2 chunks/s

nearest neighbors of git://github.com/branow/webphone/README.md#chunk-0@c6feb30927daddb0a18847819f083bf36ac30058:
  0.726  git://github.com/branow/webphone/client/README.md#chunk-0@c6feb30927daddb0a18847819f083bf36ac30058
  0.712  git://github.com/branow/webphone/README.md#chunk-1@c6feb30927daddb0a18847819f083bf36ac30058
  0.590  git://github.com/branow/webphone/client/src/hooks/useSip.ts#chunk-0@c6feb30927daddb0a18847819f083bf36ac30058
  0.570  git://github.com/branow/webphone/client/src/services/history.ts#chunk-0@c6feb30927daddb0a18847819f083bf36ac30058
  0.563  git://github.com/branow/webphone/client/src/pages/dialpad/DialPadPage.tsx#chunk-0@c6feb30927daddb0a18847819f083bf36ac30058


Step 9: first clustering pass — HDBSCAN directly on the raw 1024-dim vectors, deliberately **without** UMAP reduction, to see what density clustering does on its own before adding moving parts (expected failure mode at this dimensionality: distances concentrate, so many chunks land in no cluster at all — the noise count below is the symptom to watch).

HDBSCAN's knobs, i.e. what we'll need to figure out:
- `min_cluster_size` — the main one: smallest group that counts as a cluster. Below it, points are *noise*. For us this is effectively "minimum children per tree node".
- `min_samples` — how conservative the density estimate is; higher = stricter = more noise. Defaults to `min_cluster_size`; the usual advice is to tune `min_cluster_size` first and only touch this if noise is wrong in both directions.
- `cluster_selection_method` — `"eom"` (default) picks the most stable clusters from HDBSCAN's internal hierarchy, giving variable-size clusters; `"leaf"` cuts at the finest level, giving many small uniform clusters. For tree-building, `leaf` may actually fit better (we want small parents), worth an A/B.
- `cluster_selection_epsilon` — merges clusters closer than this distance; a lever against over-fragmentation. Scale depends on the metric, so tune last.
- `metric` — our vectors are unit-normalized, so euclidean distance is equivalent to cosine for ranking (d² = 2 − 2·cos); euclidean is the safe, supported choice.
- (design-sketch note: the min-floor/max-cap per level from `design-sketch-v1.md` is *not* an HDBSCAN parameter — it will have to be enforced by post-splitting/merging around HDBSCAN's output.)

Every chunk gets its cluster label; noise = label −1. The view prints cluster count, size distribution, the full member listing per cluster, and the earmarked empirical test: whether webphone's `uk/es/en/translation.json` chunks (near-duplicate content in three languages) collapse into one cluster.

In [9]:
from sklearn.cluster import HDBSCAN

MIN_CLUSTER_SIZE = 5

db = sqlite3.connect(DB_PATH)
rows = db.execute("SELECT hash, vector FROM embeddings WHERE model = ?", (_model_name,)).fetchall()
hashes = [r[0] for r in rows]
matrix = np.frombuffer(b"".join(r[1] for r in rows), dtype=np.float32).reshape(len(rows), -1)

labels = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, metric="euclidean").fit_predict(matrix)

# map hashes back to chunk paths (one hash can back several URIs; here it's 1:1)
paths_by_hash = defaultdict(list)
for path, chunk_no, h in db.execute("SELECT path, chunk_no, hash FROM chunks"):
    paths_by_hash[h].append(f"{path}#chunk-{chunk_no}")
db.close()

clusters = defaultdict(list)
for h, label in zip(hashes, labels):
    clusters[int(label)].extend(paths_by_hash[h])

noise = clusters.pop(-1, [])
sizes = sorted((len(m) for m in clusters.values()), reverse=True)
print(f"{len(clusters)} clusters covering {sum(sizes)} chunks; {len(noise)} chunks are noise "
      f"({100 * len(noise) / len(labels):.0f}% — high noise = distance concentration in 1024 dims)")
print(f"cluster sizes: {sizes}")

print("\n=== earmarked test: where did the translation.json chunks land? ===")
for h, label in zip(hashes, labels):
    for p in paths_by_hash[h]:
        if "locales" in p:
            print(f"  cluster {label:>3}  {p}")

print("\n=== full member listing, largest cluster first (noise last) ===")
for label, members in sorted(clusters.items(), key=lambda kv: -len(kv[1])):
    print(f"\n--- cluster {label} ({len(members)} chunks) ---")
    for p in sorted(members):
        print(f"  {p}")
print(f"\n--- noise ({len(noise)} chunks, no cluster) ---")
for p in sorted(noise):
    print(f"  {p}")

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


2 clusters covering 490 chunks; 20 chunks are noise (4% — high noise = distance concentration in 1024 dims)
cluster sizes: [481, 9]

=== earmarked test: where did the translation.json chunks land? ===
  cluster   0  branow/webphone/client/public/locales/uk/translation.json#chunk-0
  cluster   0  branow/webphone/client/public/locales/uk/translation.json#chunk-1
  cluster   0  branow/webphone/client/public/locales/uk/translation.json#chunk-2
  cluster   0  branow/webphone/client/public/locales/uk/translation.json#chunk-3
  cluster   0  branow/webphone/client/public/locales/es/translation.json#chunk-0
  cluster   0  branow/webphone/client/public/locales/es/translation.json#chunk-1
  cluster   0  branow/webphone/client/public/locales/es/translation.json#chunk-2
  cluster   0  branow/webphone/client/public/locales/es/translation.json#chunk-3
  cluster   0  branow/webphone/client/public/locales/en/translation.json#chunk-0
  cluster   0  branow/webphone/client/public/locales/en/translation.js

Step 10: UMAP → HDBSCAN. UMAP squashes the 1024-dim vectors down to `UMAP_DIMS`, keeping each point's nearest neighbors intact while letting global distances distort — turning step 9's flat plateau back into mountains and valleys HDBSCAN can see. Working range per the UMAP/HDBSCAN authors is ~5–50 dims (RAPTOR uses 10); this run tries 50 to see that end of the range. `random_state` pins UMAP's randomness so re-runs are comparable. Same view as step 9 so the two results sit side by side; success criterion for the earmarked test flips — translations should now be one cluster *separate* from the Java/TS code.

In [10]:
import sqlite3
import tomllib
from collections import defaultdict
from pathlib import Path

import numpy as np
import umap
from sklearn.cluster import HDBSCAN

# self-contained: loads everything from the DB, so this cell runs in a fresh kernel
APP_DIR = Path.cwd()
DB_PATH = APP_DIR / ".hrono" / "hrono.db"
with open(APP_DIR / "embedding.toml", "rb") as f:
    _model_name = tomllib.load(f)["model"]["name"]
MIN_CLUSTER_SIZE = 5
UMAP_DIMS = 25

db = sqlite3.connect(DB_PATH)
rows = db.execute("SELECT hash, vector FROM embeddings WHERE model = ?", (_model_name,)).fetchall()
hashes = [r[0] for r in rows]
matrix = np.frombuffer(b"".join(r[1] for r in rows), dtype=np.float32).reshape(len(rows), -1)
paths_by_hash = defaultdict(list)
for path, chunk_no, h in db.execute("SELECT path, chunk_no, hash FROM chunks"):
    paths_by_hash[h].append(f"{path}#chunk-{chunk_no}")
db.close()

reducer = umap.UMAP(n_components=UMAP_DIMS, metric="cosine", random_state=42)
reduced = reducer.fit_transform(matrix)
print(f"reduced {matrix.shape[1]} dims -> {reduced.shape[1]} dims for {reduced.shape[0]} chunks")

labels_umap = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, metric="euclidean").fit_predict(reduced)

clusters_umap = defaultdict(list)
for h, label in zip(hashes, labels_umap):
    clusters_umap[int(label)].extend(paths_by_hash[h])

noise_umap = clusters_umap.pop(-1, [])
sizes_umap = sorted((len(m) for m in clusters_umap.values()), reverse=True)
print(f"{len(clusters_umap)} clusters covering {sum(sizes_umap)} chunks; {len(noise_umap)} noise "
      f"({100 * len(noise_umap) / len(labels_umap):.0f}%)  [step 9 without UMAP: 2 clusters, sizes 481/9]")
print(f"cluster sizes: {sizes_umap}")

print("\n=== earmarked test: translations should be one cluster, separate from code ===")
for h, label in zip(hashes, labels_umap):
    for p in paths_by_hash[h]:
        if "locales" in p:
            print(f"  cluster {label:>3}  {p}")

print("\n=== full member listing, largest cluster first (noise last) ===")
for label, members in sorted(clusters_umap.items(), key=lambda kv: -len(kv[1])):
    print(f"\n--- cluster {label} ({len(members)} chunks) ---")
    for p in sorted(members):
        print(f"  {p}")
print(f"\n--- noise ({len(noise_umap)} chunks, no cluster) ---")
for p in sorted(noise_umap):
    print(f"  {p}")

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


reduced 1024 dims -> 25 dims for 510 chunks
12 clusters covering 489 chunks; 21 noise (4%)  [step 9 without UMAP: 2 clusters, sizes 481/9]
cluster sizes: [252, 78, 28, 19, 19, 18, 16, 14, 14, 12, 11, 8]

=== earmarked test: translations should be one cluster, separate from code ===
  cluster   3  branow/webphone/client/public/locales/uk/translation.json#chunk-0
  cluster   3  branow/webphone/client/public/locales/uk/translation.json#chunk-1
  cluster   3  branow/webphone/client/public/locales/uk/translation.json#chunk-2
  cluster   3  branow/webphone/client/public/locales/uk/translation.json#chunk-3
  cluster   3  branow/webphone/client/public/locales/es/translation.json#chunk-0
  cluster   3  branow/webphone/client/public/locales/es/translation.json#chunk-1
  cluster   3  branow/webphone/client/public/locales/es/translation.json#chunk-2
  cluster   3  branow/webphone/client/public/locales/es/translation.json#chunk-3
  cluster   3  branow/webphone/client/public/locales/en/translation.j

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


Step 11: same UMAP-reduced vectors, but `cluster_selection_method="leaf"` — instead of picking the most *stable* islands from HDBSCAN's internal hierarchy (`eom`, step 10), cut at the finest level: every leaf of the condensed tree becomes a cluster. Target: break up step 10's 256-chunk webphone-frontend lump into topical sub-groups. Expected trade-off: more, smaller clusters and typically more noise; the comparison to watch is what happens to cluster 2's members and whether the good clusters from step 10 (translations, per-topic server Java) survive roughly intact.

In [11]:
labels_leaf = HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE, metric="euclidean", cluster_selection_method="leaf"
).fit_predict(reduced)

clusters_leaf = defaultdict(list)
for h, label in zip(hashes, labels_leaf):
    clusters_leaf[int(label)].extend(paths_by_hash[h])

noise_leaf = clusters_leaf.pop(-1, [])
sizes_leaf = sorted((len(m) for m in clusters_leaf.values()), reverse=True)
print(f"{len(clusters_leaf)} clusters covering {sum(sizes_leaf)} chunks; {len(noise_leaf)} noise "
      f"({100 * len(noise_leaf) / len(labels_leaf):.0f}%)  [step 10 eom: 12 clusters, sizes {sizes_umap}]")
print(f"cluster sizes: {sizes_leaf}")

print("\n=== earmarked test: translations still one cluster? ===")
for h, label in zip(hashes, labels_leaf):
    for p in paths_by_hash[h]:
        if "locales" in p:
            print(f"  cluster {label:>3}  {p}")

print("\n=== full member listing, largest cluster first (noise last) ===")
for label, members in sorted(clusters_leaf.items(), key=lambda kv: -len(kv[1])):
    print(f"\n--- cluster {label} ({len(members)} chunks) ---")
    for p in sorted(members):
        print(f"  {p}")
print(f"\n--- noise ({len(noise_leaf)} chunks, no cluster) ---")
for p in sorted(noise_leaf):
    print(f"  {p}")

33 clusters covering 386 chunks; 124 noise (24%)  [step 10 eom: 12 clusters, sizes [252, 78, 28, 19, 19, 18, 16, 14, 14, 12, 11, 8]]
cluster sizes: [24, 20, 19, 19, 19, 18, 16, 15, 15, 14, 14, 14, 12, 12, 11, 11, 10, 10, 9, 9, 9, 9, 9, 9, 8, 8, 8, 7, 7, 6, 5, 5, 5]

=== earmarked test: translations still one cluster? ===
  cluster   4  branow/webphone/client/public/locales/uk/translation.json#chunk-0
  cluster   4  branow/webphone/client/public/locales/uk/translation.json#chunk-1
  cluster   4  branow/webphone/client/public/locales/uk/translation.json#chunk-2
  cluster   4  branow/webphone/client/public/locales/uk/translation.json#chunk-3
  cluster   4  branow/webphone/client/public/locales/es/translation.json#chunk-0
  cluster   4  branow/webphone/client/public/locales/es/translation.json#chunk-1
  cluster   4  branow/webphone/client/public/locales/es/translation.json#chunk-2
  cluster   4  branow/webphone/client/public/locales/es/translation.json#chunk-3
  cluster   4  branow/webphon

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


Step 12: parameter sweep instead of one-off guessing — dims ∈ {12, 25, 50} × selection ∈ {eom, leaf}, one row per config. Screening metrics per `design-sketch-v1.md` (quick geometric screening now; the real judge later is a task-based eval set):
- `clusters`, `noise_pct` — coverage vs granularity trade-off
- `max_size` / `median_size` — does the size distribution fit "small parents" tree-building (target: no mega-cluster, sizes roughly 5-30)
- `silhouette` — how geometrically separated the clusters are, computed on non-noise points in the reduced space (higher better; >0.3 decent, ~0 mushy). Caveat: silhouette favors round blobs while HDBSCAN finds arbitrary shapes, and each row's score lives in its *own* reduced space — treat as a screen, not a verdict.
- `translations_ok` — the known-group test: all 12 `locales/*/translation.json` chunks in one non-noise cluster.

Self-contained cell (loads from the DB), single UMAP seed — a follow-up worth doing before trusting a winner: repeat with 2-3 seeds to check stability.

In [12]:
import sqlite3
import tomllib
from pathlib import Path
from statistics import median

import numpy as np
import pandas as pd
import umap
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score

APP_DIR = Path.cwd()
DB_PATH = APP_DIR / ".hrono" / "hrono.db"
with open(APP_DIR / "embedding.toml", "rb") as f:
    _model_name = tomllib.load(f)["model"]["name"]
MIN_CLUSTER_SIZE = 5

db = sqlite3.connect(DB_PATH)
rows = db.execute("SELECT hash, vector FROM embeddings WHERE model = ?", (_model_name,)).fetchall()
hashes = [r[0] for r in rows]
matrix = np.frombuffer(b"".join(r[1] for r in rows), dtype=np.float32).reshape(len(rows), -1)
translation_hashes = {h for (h,) in db.execute("SELECT hash FROM chunks WHERE path LIKE '%locales%'")}
db.close()

translation_idx = [i for i, h in enumerate(hashes) if h in translation_hashes]

results = []
for dims in [12, 25, 50]:
    reduced_d = umap.UMAP(n_components=dims, metric="cosine", random_state=42).fit_transform(matrix)
    for method in ["eom", "leaf"]:
        labels_d = HDBSCAN(
            min_cluster_size=MIN_CLUSTER_SIZE, metric="euclidean", cluster_selection_method=method
        ).fit_predict(reduced_d)
        clustered = labels_d >= 0
        sizes_d = np.bincount(labels_d[clustered]) if clustered.any() else np.array([])
        t_labels = {labels_d[i] for i in translation_idx}
        results.append({
            "dims": dims,
            "method": method,
            "clusters": len(sizes_d),
            "noise_pct": round(100 * (~clustered).sum() / len(labels_d)),
            "max_size": int(sizes_d.max()) if len(sizes_d) else 0,
            "median_size": median(sizes_d.tolist()) if len(sizes_d) else 0,
            "silhouette": round(float(silhouette_score(reduced_d[clustered], labels_d[clustered])), 3)
                          if len(set(labels_d[clustered])) > 1 else float("nan"),
            "translations_ok": len(t_labels) == 1 and -1 not in t_labels,
        })

sweep = pd.DataFrame(results)
sweep

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


,dims,method,clusters,noise_pct,max_size,median_size,silhouette,translations_ok
0,12,eom,12,5,252,15.5,0.660,True
1,12,leaf,33,29,24,9.0,0.584,True
2,25,eom,12,4,252,17.0,0.699,True
3,25,leaf,33,24,24,10.0,0.556,True
4,50,eom,12,3,256,16.5,0.669,True
5,50,leaf,38,27,19,8.5,0.555,True


Step 13: path-aware similarity, per `design-sketch-v1.md`'s `α·path_similarity + (1-α)·semantic_similarity`. Implementation: embed the *path string* with the same model as content, then concatenate `[√(1-α)·content_vec ; √α·path_vec]` — for unit vectors the dot product of two such stacks is exactly `(1-α)·content_sim + α·path_sim`, so the weighted blend from the sketch falls out of plain geometry and the downstream pipeline (UMAP, HDBSCAN, ANN later) keeps working on ordinary vectors — no O(n²) custom distance matrix. Embedding the path (instead of the sketch's shared-prefix ratio) also lets semantically similar paths match across sources.

Two known hazards of the α dial: at α=1 clustering degenerates into the filesystem tree (defeating the point — clustering exists to find structure directories miss), and cross-source paths share nothing, so high α pushes toward per-source clusters. Hence a sweep, α ∈ {0, 0.15, 0.3, 0.5, 0.7}, at the step-12 winner (25 dims), both selection methods, with step 12's metrics plus `path_coherence`: the size-weighted mean share of cluster members agreeing on their 3-segment directory prefix — expected to rise with α; the interesting read is what noise/sizes/translations do as it rises.

In [13]:
import sqlite3
import tomllib
from pathlib import Path
from statistics import median

import numpy as np
import pandas as pd
import umap
from sentence_transformers import SentenceTransformer
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score

APP_DIR = Path.cwd()
DB_PATH = APP_DIR / ".hrono" / "hrono.db"
with open(APP_DIR / "embedding.toml", "rb") as f:
    _emb_cfg = tomllib.load(f)["model"]
_model_name, _batch_size = _emb_cfg["name"], _emb_cfg["batch_size"]
MIN_CLUSTER_SIZE = 5
UMAP_DIMS = 25

db = sqlite3.connect(DB_PATH)
rows = db.execute("""
    SELECT c.hash, c.path, e.vector FROM chunks c
    JOIN embeddings e ON e.hash = c.hash AND e.model = ?
    GROUP BY c.hash
""", (_model_name,)).fetchall()
db.close()

hashes = [r[0] for r in rows]
chunk_paths = [r[1] for r in rows]
content_vecs = np.frombuffer(b"".join(r[2] for r in rows), dtype=np.float32).reshape(len(rows), -1)

# embed each distinct path once with the same model (short strings - fast)
model = SentenceTransformer(_model_name)
distinct_paths = sorted(set(chunk_paths))
path_vec_by_path = dict(zip(distinct_paths, model.encode(
    distinct_paths, batch_size=_batch_size, normalize_embeddings=True, show_progress_bar=False)))
path_vecs = np.stack([path_vec_by_path[p] for p in chunk_paths])
print(f"embedded {len(distinct_paths)} distinct paths")

translation_idx = [i for i, p in enumerate(chunk_paths) if "locales" in p]
prefixes = np.array(["/".join(p.split("/")[:3]) for p in chunk_paths])


def path_coherence(labels: np.ndarray) -> float:
    """Size-weighted mean, over clusters, of the share of members whose
    3-segment path prefix matches the cluster's most common prefix."""
    agree = total = 0
    for label in set(labels) - {-1}:
        member_prefixes = prefixes[labels == label]
        counts = pd.Series(member_prefixes).value_counts()
        agree += counts.iloc[0]
        total += len(member_prefixes)
    return agree / total if total else float("nan")


results = []
for alpha in [0.0, 0.15, 0.3, 0.5, 0.7]:
    combined = np.hstack([np.sqrt(1 - alpha) * content_vecs, np.sqrt(alpha) * path_vecs])
    reduced_a = umap.UMAP(n_components=UMAP_DIMS, metric="cosine", random_state=42).fit_transform(combined)
    for method in ["eom", "leaf"]:
        labels_a = HDBSCAN(
            min_cluster_size=MIN_CLUSTER_SIZE, metric="euclidean", cluster_selection_method=method
        ).fit_predict(reduced_a)
        clustered = labels_a >= 0
        sizes_a = np.bincount(labels_a[clustered]) if clustered.any() else np.array([])
        t_labels = {labels_a[i] for i in translation_idx}
        results.append({
            "alpha": alpha,
            "method": method,
            "clusters": len(sizes_a),
            "noise_pct": round(100 * (~clustered).sum() / len(labels_a)),
            "max_size": int(sizes_a.max()) if len(sizes_a) else 0,
            "median_size": median(sizes_a.tolist()) if len(sizes_a) else 0,
            "silhouette": round(float(silhouette_score(reduced_a[clustered], labels_a[clustered])), 3)
                          if len(set(labels_a[clustered])) > 1 else float("nan"),
            "path_coherence": round(path_coherence(labels_a), 2),
            "translations_ok": len(t_labels) == 1 and -1 not in t_labels,
        })

alpha_sweep = pd.DataFrame(results)
alpha_sweep

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

embedded 482 distinct paths


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


,alpha,method,clusters,noise_pct,max_size,median_size,silhouette,path_coherence,translations_ok
0,0.00,eom,12,5,250,15.0,0.686,0.92,True
1,0.00,leaf,34,33,20,9.0,0.584,0.96,True
2,0.15,eom,14,4,249,13.5,0.699,0.92,True
3,0.15,leaf,36,25,26,9.0,0.536,0.94,True
4,0.30,eom,9,0,249,20.0,0.628,0.93,True
5,0.30,leaf,35,20,27,9.0,0.539,0.95,True
6,0.50,eom,16,3,248,17.5,0.619,0.93,True
7,0.50,leaf,37,17,28,8.0,0.598,0.95,True
8,0.70,eom,26,7,60,14.5,0.603,0.92,True
9,0.70,leaf,41,20,21,8.0,0.585,0.96,True


Step 14: determinism-constrained sweep. Decision context: incremental updates must be *re-derivable* — same content, same output — and UMAP fails that in spirit: re-fitting on slightly changed data moves every point (frozen-transform mode exists but introduces fitted state that goes stale, which is the mess this project explicitly refuses). So: which **deterministic** reduction (or none) gets closest to the UMAP results?

Candidates, strongest determinism first:
- `raw` — no reduction, cluster the 1024-dim vectors directly (step 9 showed eom collapses there; question is whether leaf + path-α survives)
- `trunc256` / `trunc128` — Matryoshka truncation: keep the first k dims of the model's output and renormalize. Qwen3-Embedding is trained for this. **Data-independent** — nothing fitted, nothing to go stale, deterministic forever.
- `rp50` — Gaussian random projection to 50 dims: a fixed seeded matrix, **data-independent**, approximately distance-preserving (Johnson-Lindenstrauss).
- `pca50` — PCA to 50 dims: deterministic given the data, but *fitted on* the data — same staleness class as UMAP, only milder. Included as reference.

All rows use `leaf` + the step-13 path-concatenation, α ∈ {0, 0.5}. Reference to beat (UMAP-25, leaf): α=0 → 34 clusters / 33% noise / max 20; α=0.5 → 37 clusters / 17% noise / max 28, coherence 0.95. Caveat: silhouette is computed in each row's own space — not comparable across reducers; judge on noise/sizes/coherence/translations.

In [14]:
import sqlite3
import tomllib
from pathlib import Path
from statistics import median

import numpy as np
import pandas as pd
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.random_projection import GaussianRandomProjection

APP_DIR = Path.cwd()
DB_PATH = APP_DIR / ".hrono" / "hrono.db"
with open(APP_DIR / "embedding.toml", "rb") as f:
    _emb_cfg = tomllib.load(f)["model"]
_model_name, _batch_size = _emb_cfg["name"], _emb_cfg["batch_size"]
MIN_CLUSTER_SIZE = 5

db = sqlite3.connect(DB_PATH)
rows = db.execute("""
    SELECT c.hash, c.path, e.vector FROM chunks c
    JOIN embeddings e ON e.hash = c.hash AND e.model = ?
    GROUP BY c.hash
""", (_model_name,)).fetchall()
db.close()

chunk_paths = [r[1] for r in rows]
content_vecs = np.frombuffer(b"".join(r[2] for r in rows), dtype=np.float32).reshape(len(rows), -1)

# path vectors: reuse from step 13 if the kernel has them, else re-embed
try:
    path_vecs
except NameError:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(_model_name)
    distinct_paths = sorted(set(chunk_paths))
    path_vec_by_path = dict(zip(distinct_paths, model.encode(
        distinct_paths, batch_size=_batch_size, normalize_embeddings=True, show_progress_bar=False)))
    path_vecs = np.stack([path_vec_by_path[p] for p in chunk_paths])

translation_idx = [i for i, p in enumerate(chunk_paths) if "locales" in p]
prefixes = np.array(["/".join(p.split("/")[:3]) for p in chunk_paths])


def path_coherence(labels: np.ndarray) -> float:
    agree = total = 0
    for label in set(labels) - {-1}:
        counts = pd.Series(prefixes[labels == label]).value_counts()
        agree += counts.iloc[0]
        total += int((labels == label).sum())
    return agree / total if total else float("nan")


def truncate(vecs: np.ndarray, k: int) -> np.ndarray:
    """Matryoshka truncation: first k dims, renormalized to unit length."""
    cut = vecs[:, :k]
    return cut / np.linalg.norm(cut, axis=1, keepdims=True)


REDUCERS = {
    "raw": lambda X: X,
    "trunc256": None,  # handled before concatenation, see below
    "trunc128": None,
    "rp50": lambda X: GaussianRandomProjection(n_components=50, random_state=42).fit_transform(X),
    "pca50": lambda X: PCA(n_components=50, random_state=42).fit_transform(X),
}

results = []
for name in REDUCERS:
    for alpha in [0.0, 0.5]:
        if name.startswith("trunc"):
            k = int(name.removeprefix("trunc"))
            c, p = truncate(content_vecs, k), truncate(path_vecs, k)
        else:
            c, p = content_vecs, path_vecs
        combined = np.hstack([np.sqrt(1 - alpha) * c, np.sqrt(alpha) * p])
        reduced_r = combined if name.startswith("trunc") else REDUCERS[name](combined)
        labels_r = HDBSCAN(
            min_cluster_size=MIN_CLUSTER_SIZE, metric="euclidean", cluster_selection_method="leaf"
        ).fit_predict(reduced_r)
        clustered = labels_r >= 0
        sizes_r = np.bincount(labels_r[clustered]) if clustered.any() else np.array([])
        t_labels = {labels_r[i] for i in translation_idx}
        results.append({
            "reducer": name,
            "alpha": alpha,
            "clusters": len(sizes_r),
            "noise_pct": round(100 * (~clustered).sum() / len(labels_r)),
            "max_size": int(sizes_r.max()) if len(sizes_r) else 0,
            "median_size": median(sizes_r.tolist()) if len(sizes_r) else 0,
            "silhouette": round(float(silhouette_score(reduced_r[clustered], labels_r[clustered])), 3)
                          if len(set(labels_r[clustered])) > 1 else float("nan"),
            "path_coherence": round(path_coherence(labels_r), 2),
            "translations_ok": len(t_labels) == 1 and -1 not in t_labels,
        })

det_sweep = pd.DataFrame(results)
det_sweep

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `cop

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


,reducer,alpha,clusters,noise_pct,max_size,median_size,silhouette,path_coherence,translations_ok
0,raw,0.0,13,66,59,10.0,0.200,0.96,False
1,raw,0.5,14,68,31,11.0,0.235,0.99,True
2,trunc256,0.0,15,68,39,8.0,0.240,0.96,False
3,trunc256,0.5,19,60,70,6.0,0.218,0.98,True
4,trunc128,0.0,14,66,38,8.5,0.216,0.98,False
5,trunc128,0.5,19,66,30,7.0,0.261,0.98,True
6,rp50,0.0,12,79,17,9.0,0.214,0.94,False
7,rp50,0.5,14,74,17,8.0,0.268,0.99,True
8,pca50,0.0,21,59,22,8.0,0.305,0.97,True
9,pca50,0.5,23,51,20,10.0,0.307,0.96,True


Step 15: stability-under-drift experiment — the user's core worry about UMAP quantified. A re-fit on slightly different data is allowed to move coordinates; the question is whether the *partition* survives. Baseline: full corpus, winning recipe (UMAP-25 + HDBSCAN leaf, path-α 0.5, fixed seed). Two drift scenarios, each re-running the FULL pipeline from scratch:
- `pr_merge` — ~5% of chunks removed at random (a merged PR changing/deleting content)
- `new_repo` — baseline computed *without* football-match-mvc, then the full corpus (a repo being added)

Metrics over the chunks present in both runs:
- `ARI` (adjusted Rand index) — pairwise agreement of the two partitions; 1.0 = identical grouping, 0 = random relabeling
- `cluster_survival` — size-weighted share of each baseline cluster's members that stay together (with the cluster's majority) in the new run — the "does it read as an extension or a reshuffle" number
- `noise_flips` — share of common chunks that changed noise status either way

In [15]:
import sqlite3
import tomllib
from pathlib import Path

import numpy as np
import pandas as pd
import umap
from sklearn.cluster import HDBSCAN
from sklearn.metrics import adjusted_rand_score

APP_DIR = Path.cwd()
DB_PATH = APP_DIR / ".hrono" / "hrono.db"
with open(APP_DIR / "embedding.toml", "rb") as f:
    _emb_cfg = tomllib.load(f)["model"]
_model_name, _batch_size = _emb_cfg["name"], _emb_cfg["batch_size"]
MIN_CLUSTER_SIZE = 5
UMAP_DIMS = 25
ALPHA = 0.5

db = sqlite3.connect(DB_PATH)
rows = db.execute("""
    SELECT c.hash, c.path, e.vector FROM chunks c
    JOIN embeddings e ON e.hash = c.hash AND e.model = ?
    GROUP BY c.hash
""", (_model_name,)).fetchall()
db.close()

all_paths = [r[1] for r in rows]
content_vecs = np.frombuffer(b"".join(r[2] for r in rows), dtype=np.float32).reshape(len(rows), -1)

try:
    path_vec_by_path
except NameError:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(_model_name)
    distinct_paths = sorted(set(all_paths))
    path_vec_by_path = dict(zip(distinct_paths, model.encode(
        distinct_paths, batch_size=_batch_size, normalize_embeddings=True, show_progress_bar=False)))
path_vecs = np.stack([path_vec_by_path[p] for p in all_paths])

combined_all = np.hstack([np.sqrt(1 - ALPHA) * content_vecs, np.sqrt(ALPHA) * path_vecs])


def cluster(idx: np.ndarray) -> np.ndarray:
    """Full pipeline from scratch on the subset idx: UMAP re-fit + HDBSCAN leaf."""
    reduced_s = umap.UMAP(n_components=UMAP_DIMS, metric="cosine", random_state=42).fit_transform(combined_all[idx])
    return HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, metric="euclidean",
                   cluster_selection_method="leaf").fit_predict(reduced_s)


def compare(idx_a: np.ndarray, labels_a: np.ndarray, idx_b: np.ndarray, labels_b: np.ndarray) -> dict:
    """Stability metrics over chunks present in both runs."""
    pos_a = {g: i for i, g in enumerate(idx_a)}
    pos_b = {g: i for i, g in enumerate(idx_b)}
    common = [g for g in idx_a if g in pos_b]
    la = np.array([labels_a[pos_a[g]] for g in common])
    lb = np.array([labels_b[pos_b[g]] for g in common])
    survival_num = survival_den = 0
    for c in set(la) - {-1}:
        in_c = lb[la == c]
        counts = pd.Series(in_c).value_counts()
        survival_num += int(counts.iloc[0])
        survival_den += len(in_c)
    return {
        "common_chunks": len(common),
        "ARI": round(adjusted_rand_score(la, lb), 3),
        "cluster_survival": round(survival_num / survival_den, 2) if survival_den else float("nan"),
        "noise_flips_pct": round(100 * ((la == -1) != (lb == -1)).mean()),
    }


rng = np.random.default_rng(7)
n = len(rows)
idx_full = np.arange(n)
labels_full = cluster(idx_full)

# scenario 1: merged PR — 5% of chunks removed
idx_pr = np.sort(rng.choice(n, size=int(n * 0.95), replace=False))
labels_pr = cluster(idx_pr)

# scenario 2: repo added — baseline lacks football-match-mvc
idx_nofmm = np.array([i for i, p in enumerate(all_paths) if "football-match-mvc" not in p])
labels_nofmm = cluster(idx_nofmm)

stability = pd.DataFrame([
    {"scenario": "pr_merge (5% removed vs full)", **compare(idx_full, labels_full, idx_pr, labels_pr)},
    {"scenario": "new_repo (without fmm vs full)", **compare(idx_nofmm, labels_nofmm, idx_full, labels_full)},
])
stability

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


,scenario,common_chunks,ARI,cluster_survival,noise_flips_pct
0,pr_merge (5% removed vs full),484,0.464,0.85,16
1,new_repo (without fmm vs full),419,0.486,0.81,15


Step 16: warm-started UMAP — the stability fix under test. With a fixed seed UMAP is deterministic but not *continuous*: small input changes can re-solve the whole layout. Warm start passes the previous run's coordinates as `init`: surviving chunks start where they ended last time, new chunks start at their nearest existing neighbor's position (plus tiny seeded jitter so they aren't exactly stacked), and the optimizer only adjusts what the new data forces. Same two drift scenarios and metrics as step 15; the table shows cold vs warm side by side. If warm ARI/survival climb toward 1.0, rebuilds become "extensions" and the churn objection to UMAP mostly dissolves; requires persisting the coordinates alongside the tree (they become part of stored state — a deliberate trade against the pure re-derive-from-content philosophy, worth it only if the numbers say so).

In [16]:
# depends on step 15 (combined_all, compare, idx_*, stability) — run that cell first
def fit(idx: np.ndarray, init="spectral"):
    reducer = umap.UMAP(n_components=UMAP_DIMS, metric="cosine", random_state=42, init=init)
    coords = reducer.fit_transform(combined_all[idx])
    labels = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, metric="euclidean",
                     cluster_selection_method="leaf").fit_predict(coords)
    return coords, labels


coords_full, labels_full_c = fit(idx_full)
coords_nofmm, labels_nofmm_c = fit(idx_nofmm)
rng16 = np.random.default_rng(16)

# scenario 1 warm: pr_merge — every surviving chunk starts at its previous coordinates
_, labels_pr_warm = fit(idx_pr, init=coords_full[idx_pr].copy())

# scenario 2 warm: new_repo — shared chunks keep coords, new (fmm) chunks start at nearest baseline neighbor
pos_nofmm = {g: i for i, g in enumerate(idx_nofmm)}
init_full = np.zeros((len(idx_full), UMAP_DIMS), dtype=np.float32)
new_ids = np.array([g for g in idx_full if g not in pos_nofmm])
for j, g in enumerate(idx_full):
    if g in pos_nofmm:
        init_full[j] = coords_nofmm[pos_nofmm[g]]
sims16 = combined_all[new_ids] @ combined_all[idx_nofmm].T
init_full[new_ids] = coords_nofmm[sims16.argmax(axis=1)] + rng16.normal(0, 0.05, (len(new_ids), UMAP_DIMS))
_, labels_full_warm = fit(idx_full, init=init_full)

warm = pd.DataFrame([
    {"scenario": "pr_merge", "mode": "warm", **compare(idx_full, labels_full_c, idx_pr, labels_pr_warm)},
    {"scenario": "new_repo", "mode": "warm", **compare(idx_nofmm, labels_nofmm_c, idx_full, labels_full_warm)},
])
cold = stability.assign(mode="cold").replace({
    "pr_merge (5% removed vs full)": "pr_merge", "new_repo (without fmm vs full)": "new_repo"})
pd.concat([cold, warm]).sort_values(["scenario", "mode"])[
    ["scenario", "mode", "common_chunks", "ARI", "cluster_survival", "noise_flips_pct"]
].reset_index(drop=True)

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


,scenario,mode,common_chunks,ARI,cluster_survival,noise_flips_pct
0,new_repo,cold,419,0.486,0.81,15
1,new_repo,warm,419,0.589,0.86,11
2,pr_merge,cold,484,0.464,0.85,16
3,pr_merge,warm,484,0.550,0.92,14


Step 17: the drift visualized — a side-by-side diff of cluster state before vs after the `pr_merge` drift (5% of chunks removed, warm rebuild), so the churn numbers become inspectable chunk by chunk. After-clusters are matched to before-clusters by member overlap; a before-cluster that split shows several after-columns. Color legend:
- <span style="color:#333">black — stayed with its group (the desired outcome)</span>
- <span style="color:#b91c1c;text-decoration:line-through">red struck — removed by the simulated PR (legitimate change)</span>
- <span style="color:#d97706;font-weight:bold">orange — content UNCHANGED but moved to a different cluster: the illegitimate churn each rebuild costs</span>
- <span style="color:#7c3aed">purple — clustered before, noise after</span>
- <span style="color:#15803d">green — noise before, adopted into a cluster after</span>
- <span style="color:#9ca3af;font-style:italic">gray italic — noise in both runs</span>

In [17]:
# scenario: pr_merge warm — before: full corpus baseline; after: 95% subset, warm rebuild (step 16)
before_idx, before_labels = idx_full, labels_full_c
after_idx, after_labels = idx_pr, labels_pr_warm
removed_ids = set(idx_full.tolist()) - set(idx_pr.tolist())

pos_b = {g: i for i, g in enumerate(before_idx)}
pos_a = {g: i for i, g in enumerate(after_idx)}

overlap = defaultdict(Counter)
for g in after_idx:
    la, lb = after_labels[pos_a[g]], before_labels[pos_b[g]]
    if la >= 0:
        overlap[int(la)][int(lb)] += 1
after_to_before = {la: cnt.most_common(1)[0][0] for la, cnt in overlap.items()}
before_to_after = defaultdict(list)
for la, lb in sorted(after_to_before.items()):
    before_to_after[lb].append(la)

status = {}
for g in before_idx:
    lb = int(before_labels[pos_b[g]])
    if g in removed_ids:
        status[g] = "removed"
        continue
    la = int(after_labels[pos_a[g]])
    if lb == -1:
        status[g] = "noise_adopted" if la >= 0 else "noise_still"
    elif la == -1:
        status[g] = "dropped_to_noise"
    elif after_to_before.get(la) == lb:
        status[g] = "stayed"
    else:
        status[g] = "moved"

counts = Counter(status.values())
print("LEGEND (mark, count, meaning)")
print(f"  =  {counts['stayed']:>3}  stayed with its group (the desired outcome)")
print(f"  -  {counts['removed']:>3}  removed by the simulated PR (legitimate change)")
print(f"  !  {counts['moved']:>3}  content UNCHANGED but moved to another cluster (illegitimate churn)")
print(f"  ~  {counts['dropped_to_noise']:>3}  was clustered, fell to noise")
print(f"  +  {counts['noise_adopted']:>3}  was noise, adopted into a cluster")
print(f"  ?  {counts['noise_still']:>3}  noise in both runs")
print("B# = cluster id before, A# = after (matched by shared members); SPLIT = one B became several A")

MARK = {"stayed": "=", "removed": "-", "moved": "!", "dropped_to_noise": "~",
        "noise_adopted": "+", "noise_still": "?"}

before_clusters = defaultdict(list)
for g in before_idx:
    before_clusters[int(before_labels[pos_b[g]])].append(g)

for lb, members in sorted(((l, m) for l, m in before_clusters.items() if l >= 0), key=lambda kv: -len(kv[1])):
    afters = before_to_after.get(lb, [])
    after_names = ", ".join(f"A{a}" for a in afters) or "(none)"
    split = "  SPLIT" if len(afters) > 1 else ""
    print(f"\n=== B{lb} ({len(members)} chunks) -> {after_names}{split} ===")
    for g in sorted(members, key=lambda g: all_paths[g]):
        note = ""
        if status[g] == "moved":
            note = f"  -> A{int(after_labels[pos_a[g]])}"
        elif status[g] == "dropped_to_noise":
            note = "  -> noise"
        print(f"  {MARK[status[g]]} {all_paths[g]}{note}")
    arrivals = [g for la in afters for g in after_idx
                if int(after_labels[pos_a[g]]) == la and int(before_labels[pos_b[g]]) != lb]
    if arrivals:
        print("  arrivals:")
        for g in sorted(arrivals, key=lambda g: all_paths[g]):
            src = int(before_labels[pos_b[g]])
            origin = "was noise" if src == -1 else f"from B{src}"
            print(f"  + {all_paths[g]}  ({origin})")

noise_members = before_clusters.get(-1, [])
print(f"\n=== before: noise ({len(noise_members)} chunks) ===")
for g in sorted(noise_members, key=lambda g: all_paths[g]):
    note = f"  -> A{int(after_labels[pos_a[g]])}" if status[g] == "noise_adopted" else ""
    print(f"  {MARK[status[g]]} {all_paths[g]}{note}")

LEGEND (mark, count, meaning)
  =  334  stayed with its group (the desired outcome)
  -   26  removed by the simulated PR (legitimate change)
  !   37  content UNCHANGED but moved to another cluster (illegitimate churn)
  ~   30  was clustered, fell to noise
  +   36  was noise, adopted into a cluster
  ?   47  noise in both runs
B# = cluster id before, A# = after (matched by shared members); SPLIT = one B became several A

=== B4 (28 chunks) -> A1 ===
  = branow/football-match-mvc/src/main/resources/templates/auth-layout.html
  = branow/football-match-mvc/src/main/resources/templates/auth/change-password.html
  = branow/football-match-mvc/src/main/resources/templates/auth/login.html
  = branow/football-match-mvc/src/main/resources/templates/auth/logout-confirmation.html
  = branow/football-match-mvc/src/main/resources/templates/auth/register.html
  = branow/football-match-mvc/src/main/resources/templates/dashboard.html
  = branow/football-match-mvc/src/main/resources/templates/error/4

Step 18: the noise strategy. Finding from the step-17 diff: churn concentrates at the noise boundary, and "noise" is really the corpus's most distinctive chunks (no ~`min_cluster_size` similar siblings) — content that must end up in the tree, not garbage. Two-part experiment:
1. `min_cluster_size` sweep {2, 3, 5}: how much noise does a lower floor absorb, at what fragmentation cost (2 is expected to degenerate into micro-pairs).
2. Deterministic assignment: with the chosen floor, every remaining noise chunk attaches to the cluster containing its most-similar member (pure distance in the reduced space — stable, no density threshold to flutter). The listing shows each attachment with its similarity so the sanity is inspectable; a low-similarity tail would argue for a distance cap ("attach to parent node instead").

In [18]:
# depends on step 15/16 kernel state (combined_all, idx_full, all_paths)
reduced_full = umap.UMAP(n_components=UMAP_DIMS, metric="cosine", random_state=42).fit_transform(combined_all)

print("=== min_cluster_size sweep (leaf, alpha=0.5, UMAP-25) ===")
mcs_results = []
for mcs in [2, 3, 5]:
    labels_m = HDBSCAN(min_cluster_size=mcs, metric="euclidean",
                       cluster_selection_method="leaf").fit_predict(reduced_full)
    clustered_m = labels_m >= 0
    sizes_m = np.bincount(labels_m[clustered_m])
    mcs_results.append({
        "min_cluster_size": mcs,
        "clusters": len(sizes_m),
        "noise_pct": round(100 * (~clustered_m).mean()),
        "clusters_of_2_or_3": int(((sizes_m == 2) | (sizes_m == 3)).sum()),
        "max_size": int(sizes_m.max()),
        "median_size": float(np.median(sizes_m)),
    })
display(pd.DataFrame(mcs_results))

# --- part 2: deterministic nearest-cluster assignment at mcs=5 ---
labels_a5 = HDBSCAN(min_cluster_size=5, metric="euclidean",
                    cluster_selection_method="leaf").fit_predict(reduced_full)
noise_idx5 = np.where(labels_a5 == -1)[0]
cluster_idx5 = np.where(labels_a5 >= 0)[0]

# nearest = the cluster containing the most similar clustered chunk (in original combined space,
# so the assignment is independent of the reducer — deterministic given the vectors)
sims_nc = combined_all[noise_idx5] @ combined_all[cluster_idx5].T
nearest_member = sims_nc.argmax(axis=1)
assigned_cluster = labels_a5[cluster_idx5[nearest_member]]
assigned_sim = sims_nc.max(axis=1)

print(f"\n=== assignment of {len(noise_idx5)} noise chunks (mcs=5) ===")
print(f"similarity to attachment point: min {assigned_sim.min():.2f}, "
      f"median {np.median(assigned_sim):.2f}, max {assigned_sim.max():.2f}")
order = np.argsort(assigned_sim)
print("\nweakest 10 attachments (candidates for a distance cap):")
for j in order[:10]:
    g = noise_idx5[j]
    target = cluster_idx5[nearest_member[j]]
    print(f"  {assigned_sim[j]:.2f}  {all_paths[g]}  ->  A{assigned_cluster[j]} (nearest member: {all_paths[target]})")
print("\nstrongest 10 attachments:")
for j in order[::-1][:10]:
    g = noise_idx5[j]
    target = cluster_idx5[nearest_member[j]]
    print(f"  {assigned_sim[j]:.2f}  {all_paths[g]}  ->  A{assigned_cluster[j]} (nearest member: {all_paths[target]})")

full_labels = labels_a5.copy()
full_labels[noise_idx5] = assigned_cluster
final_sizes = np.bincount(full_labels)
print(f"\nafter assignment: 0% noise, {len(final_sizes)} clusters, "
      f"sizes min {final_sizes.min()} / median {np.median(final_sizes):.0f} / max {final_sizes.max()}")

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


=== min_cluster_size sweep (leaf, alpha=0.5, UMAP-25) ===


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


,min_cluster_size,clusters,noise_pct,clusters_of_2_or_3,max_size,median_size
0,2,145,13,108,7,3.0
1,3,78,22,21,12,5.0
2,5,37,17,0,28,8.0



=== assignment of 85 noise chunks (mcs=5) ===
similarity to attachment point: min 0.57, median 0.78, max 0.97

weakest 10 attachments (candidates for a distance cap):
  0.57  branow/webphone/server/src/test/java/com/scisbo/webphone/common/mongodb/EmbeddedMongoDbConfig.java  ->  A6 (nearest member: branow/webphone/server/src/test/java/com/scisbo/webphone/repositories/PhotoRepositoryIT.java)
  0.57  branow/webphone/client/src/util/identifier.ts  ->  A15 (nearest member: branow/webphone/client/src/util/format.ts)
  0.59  branow/webphone/server/src/test/java/com/scisbo/webphone/common/mongodb/EmbeddedMongoDb.java  ->  A6 (nearest member: branow/webphone/server/src/test/java/com/scisbo/webphone/repositories/PhotoRepositoryIT.java)
  0.59  branow/webphone/client/src/lib/storage.ts  ->  A35 (nearest member: branow/webphone/client/src/vite-env.d.ts)
  0.62  branow/webphone/client/src/components/call/DurationInMs.tsx  ->  A25 (nearest member: branow/webphone/client/src/components/history/Statu

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


Step 19: does zero-noise assignment actually stabilize drift? Steps 15-16 measured churn with noise as a category (16%/14% of chunks flipped noise status per rebuild). Assignment abolishes the category — but that could either absorb the flutter (attachment is a deterministic argmax over vectors) or just relabel it as cluster-to-cluster movement. Same two drift scenarios, warm-started, but the compared partitions are now *total* (leaf mcs=5 + nearest-member assignment): every common chunk has a parent in both runs, `noise_flips` is structurally 0, and `ARI`/`cluster_survival` measure the whole corpus, not just the dense cores. Comparison row `step-16 warm (cores only)` included for reference.

In [19]:
def full_partition(idx: np.ndarray, init="spectral") -> np.ndarray:
    """The settled recipe end-to-end: warm-startable UMAP -> HDBSCAN leaf -> nearest-member assignment."""
    reducer = umap.UMAP(n_components=UMAP_DIMS, metric="cosine", random_state=42, init=init)
    coords = reducer.fit_transform(combined_all[idx])
    labels = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, metric="euclidean",
                     cluster_selection_method="leaf").fit_predict(coords)
    noise_pos = np.where(labels == -1)[0]
    core_pos = np.where(labels >= 0)[0]
    if len(noise_pos):
        sims_na = combined_all[idx[noise_pos]] @ combined_all[idx[core_pos]].T
        labels[noise_pos] = labels[core_pos[sims_na.argmax(axis=1)]]
    return coords, labels


coords_full_t, labels_full_t = full_partition(idx_full)
coords_nofmm_t, labels_nofmm_t = full_partition(idx_nofmm)

# warm drifts, as in step 16
_, labels_pr_t = full_partition(idx_pr, init=coords_full_t[idx_pr].copy())

pos_nofmm_t = {g: i for i, g in enumerate(idx_nofmm)}
init_full_t = np.zeros((len(idx_full), UMAP_DIMS), dtype=np.float32)
new_ids_t = np.array([g for g in idx_full if g not in pos_nofmm_t])
for j, g in enumerate(idx_full):
    if g in pos_nofmm_t:
        init_full_t[j] = coords_nofmm_t[pos_nofmm_t[g]]
rng19 = np.random.default_rng(16)
sims19 = combined_all[new_ids_t] @ combined_all[idx_nofmm].T
init_full_t[new_ids_t] = coords_nofmm_t[sims19.argmax(axis=1)] + rng19.normal(0, 0.05, (len(new_ids_t), UMAP_DIMS))
_, labels_full_warm_t = full_partition(idx_full, init=init_full_t)

total = pd.DataFrame([
    {"scenario": "pr_merge", "partition": "total (with assignment)",
     **compare(idx_full, labels_full_t, idx_pr, labels_pr_t)},
    {"scenario": "new_repo", "partition": "total (with assignment)",
     **compare(idx_nofmm, labels_nofmm_t, idx_full, labels_full_warm_t)},
    {"scenario": "pr_merge", "partition": "step-16 warm (cores only)",
     "common_chunks": 484, "ARI": 0.550, "cluster_survival": 0.92, "noise_flips_pct": 14},
    {"scenario": "new_repo", "partition": "step-16 warm (cores only)",
     "common_chunks": 419, "ARI": 0.589, "cluster_survival": 0.86, "noise_flips_pct": 11},
]).sort_values(["scenario", "partition"]).reset_index(drop=True)
total

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


,scenario,partition,common_chunks,ARI,cluster_survival,noise_flips_pct
0,new_repo,step-16 warm (cores only),419,0.589,0.86,11
1,new_repo,total (with assignment),419,0.815,0.86,0
2,pr_merge,step-16 warm (cores only),484,0.550,0.92,14
3,pr_merge,total (with assignment),484,0.805,0.92,0


Step 20: anatomy of the mistaken moves. For both drift scenarios (warm rebuilds), every chunk whose content did not change but whose cluster did: which file, from which cluster to which (clusters named by their dominant path prefix), and whether the move is a *split fragment* (its new cluster is largely made of its old clustermates — the group divided, membership mostly preserved) or a *jump* (landed among strangers — the bad kind). This is the churn the incremental path (step 21) must eliminate.

In [20]:
def cluster_name(member_ids):
    segs = Counter("/".join(all_paths[g].split("/")[1:4]) for g in member_ids)
    top, n = segs.most_common(1)[0]
    return f"{top} [{n}/{len(member_ids)}]"


def analyze_moves(idx_before, labels_before, idx_after, labels_after, scenario):
    pos_b = {g: i for i, g in enumerate(idx_before)}
    pos_a = {g: i for i, g in enumerate(idx_after)}
    members_b, members_a = defaultdict(list), defaultdict(list)
    for g in idx_before:
        members_b[int(labels_before[pos_b[g]])].append(g)
    for g in idx_after:
        members_a[int(labels_after[pos_a[g]])].append(g)
    overlap = defaultdict(Counter)
    for g in idx_after:
        if g in pos_b and labels_after[pos_a[g]] >= 0:
            overlap[int(labels_after[pos_a[g]])][int(labels_before[pos_b[g]])] += 1
    a2b = {la: cnt.most_common(1)[0][0] for la, cnt in overlap.items()}
    names_b = {l: cluster_name(m) for l, m in members_b.items() if l >= 0}
    names_a = {l: cluster_name(m) for l, m in members_a.items() if l >= 0}
    moves = []
    for g in idx_before:
        if g not in pos_a:
            continue
        lb, la = int(labels_before[pos_b[g]]), int(labels_after[pos_a[g]])
        if lb < 0 or la < 0 or a2b.get(la) == lb:
            continue
        same_origin = sum(1 for m in members_a[la] if m in pos_b and int(labels_before[pos_b[m]]) == lb)
        kind = "split-fragment" if same_origin / len(members_a[la]) >= 0.3 else "JUMP"
        moves.append({
            "scenario": scenario, "kind": kind, "path": all_paths[g],
            "from": f"B{lb} {names_b[lb]}", "to": f"A{la} {names_a.get(la, '?')}",
        })
    return pd.DataFrame(moves)


moves_pr = analyze_moves(idx_full, labels_full_c, idx_pr, labels_pr_warm, "pr_merge")
moves_nr = analyze_moves(idx_nofmm, labels_nofmm_c, idx_full, labels_full_warm, "new_repo")
all_moves = pd.concat([moves_pr, moves_nr])
print("=== mistaken moves (content unchanged, cluster changed), by kind ===")
print(all_moves.groupby(["scenario", "kind"]).size().to_string())
print("\n=== per cluster-pair (count / from -> to) ===")
pair_counts = all_moves.groupby(["scenario", "kind", "from", "to"]).size().reset_index(name="n")
for _, r in pair_counts.sort_values("n", ascending=False).iterrows():
    print(f"  {r['n']:>2}  {r['scenario']:<9} {r['kind']:<14} {r['from']}  ->  {r['to']}")
print("\n=== every moved file ===")
for _, r in all_moves.sort_values(["scenario", "kind", "from"]).iterrows():
    print(f"  [{r['scenario']} {r['kind']}] {r['path']}")
    print(f"      {r['from']}  ->  {r['to']}")

=== mistaken moves (content unchanged, cluster changed), by kind ===
scenario  kind          
new_repo  JUMP               6
          split-fragment     5
pr_merge  JUMP              14
          split-fragment    23

=== per cluster-pair (count / from -> to) ===
   6  pr_merge  split-fragment B34 webphone/client/src [6/6]  ->  A28 webphone/client/src [20/20]
   6  pr_merge  split-fragment B28 webphone/server/src [6/6]  ->  A17 webphone/server/src [17/17]
   6  pr_merge  split-fragment B26 webphone/client/src [6/6]  ->  A15 webphone/client/src [15/15]
   5  pr_merge  split-fragment B17 webphone/server/src [5/5]  ->  A11 webphone/server/src [15/15]
   5  pr_merge  JUMP           B33 webphone/client/src [5/5]  ->  A28 webphone/client/src [20/20]
   3  pr_merge  JUMP           B32 webphone/client/src [6/6]  ->  A27 webphone/client/src [14/14]
   3  new_repo  split-fragment B23 webphone/client/src [22/22]  ->  A30 webphone/client/src [10/10]
   2  new_repo  JUMP           B10 webphone/cli

Step 21: the incremental update path — the experiment that matters. Rule: **updates never re-run the pipeline.** Existing clusters are frozen; a new/changed chunk is assigned to its nearest existing cluster if its best similarity clears a threshold `TAU`, otherwise it goes to a *novel pool*; the novel pool is clustered among itself (deterministic agglomerative, cosine, average linkage, distance threshold 0.3 ≈ sim 0.7) to form new clusters. Consequences by construction: untouched chunks NEVER move (churn = 0 — the actual project requirement); a genuinely new topic (the added repo) forms its own new subtree instead of polluting existing clusters.

Scenario: base = corpus without football-match-mvc (frozen total partition), drift = fmm's chunks arrive. Sweep TAU ∈ {0.65, 0.70, 0.75}; metrics: how many new chunks were absorbed into existing clusters vs formed new ones, sizes of the new clusters, and quality = ARI between the incremental placement of fmm chunks and where a cold full re-cluster would have put them (the expensive "ideal" reference). High ARI = the cheap deterministic path reproduces the expensive rebuild's opinion of the new content.

In [21]:
from sklearn.cluster import AgglomerativeClustering

fmm_set = set(idx_nofmm.tolist())
fmm_ids = np.array([g for g in idx_full if g not in fmm_set])

# frozen base: total partition of the no-fmm corpus (step 19)
base_labels_by_id = {g: int(labels_nofmm_t[i]) for i, g in enumerate(idx_nofmm)}

sims_new = combined_all[fmm_ids] @ combined_all[idx_nofmm].T
best_sim = sims_new.max(axis=1)
best_base = idx_nofmm[sims_new.argmax(axis=1)]

# reference: where the cold full rebuild put the fmm chunks (step 19 total partition)
pos_full = {g: i for i, g in enumerate(idx_full)}
ref_fmm = np.array([int(labels_full_t[pos_full[g]]) for g in fmm_ids])

print(f"{len(fmm_ids)} new chunks (fmm repo); best similarity to existing corpus: "
      f"min {best_sim.min():.2f} / median {np.median(best_sim):.2f} / max {best_sim.max():.2f}\n")

for tau in [0.65, 0.70, 0.75]:
    absorbed = best_sim >= tau
    novel = fmm_ids[~absorbed]
    inc_labels = np.empty(len(fmm_ids), dtype=object)
    inc_labels[absorbed] = [f"EXIST_{base_labels_by_id[g]}" for g in best_base[np.where(absorbed)[0]]]
    if len(novel) >= 2:
        agg = AgglomerativeClustering(n_clusters=None, distance_threshold=0.3,
                                      metric="cosine", linkage="average").fit(combined_all[novel])
        novel_sizes = np.bincount(agg.labels_)
        pos_novel = {g: i for i, g in enumerate(novel)}
        inc_labels[~absorbed] = [f"NEW_{agg.labels_[pos_novel[g]]}" for g in novel]
    else:
        novel_sizes = np.array([], dtype=int)
        inc_labels[~absorbed] = "NEW_0"
    from sklearn.metrics import adjusted_rand_score
    ari = adjusted_rand_score(ref_fmm, pd.factorize(inc_labels)[0])
    print(f"TAU={tau}: absorbed into existing: {int(absorbed.sum())}, novel: {len(novel)}, "
          f"new clusters: {len(novel_sizes)} (sizes {sorted(novel_sizes.tolist(), reverse=True)}), "
          f"ARI vs cold rebuild: {ari:.3f}")
    print(f"        existing-chunk churn: 0 by construction ({len(idx_nofmm)} chunks untouched)")

# name the new clusters at TAU=0.70 for inspection
tau = 0.70
absorbed = best_sim >= tau
novel = fmm_ids[~absorbed]
agg = AgglomerativeClustering(n_clusters=None, distance_threshold=0.3,
                              metric="cosine", linkage="average").fit(combined_all[novel])
print(f"\n=== new clusters formed at TAU={tau} (named by dominant prefix) ===")
for l in range(agg.labels_.max() + 1):
    members = novel[agg.labels_ == l]
    print(f"  NEW_{l} ({len(members)}): {cluster_name(members)}")
print(f"\n=== chunks absorbed into existing clusters at TAU={tau} ===")
for j in np.where(absorbed)[0]:
    g, b = fmm_ids[j], best_base[j]
    print(f"  {best_sim[j]:.2f}  {all_paths[g]}  ->  existing B{base_labels_by_id[b]} (nearest: {all_paths[b]})")

91 new chunks (fmm repo); best similarity to existing corpus: min 0.35 / median 0.53 / max 0.72

TAU=0.65: absorbed into existing: 9, novel: 82, new clusters: 27 (sizes [18, 5, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1]), ARI vs cold rebuild: 0.313
        existing-chunk churn: 0 by construction (419 chunks untouched)
TAU=0.7: absorbed into existing: 1, novel: 90, new clusters: 31 (sizes [18, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1]), ARI vs cold rebuild: 0.312
        existing-chunk churn: 0 by construction (419 chunks untouched)
TAU=0.75: absorbed into existing: 0, novel: 91, new clusters: 31 (sizes [18, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1]), ARI vs cold rebuild: 0.313
        existing-chunk churn: 0 by construction (419 chunks untouched)

=== new clusters formed at TAU=0.7 (named by dominant prefix) ===
  NEW_0 (18): football-match-mvc/src/main [18/18

Step 22: fixing the novel-pool clustering. Step 21 proved the incremental frame (0 existing churn, correct refusal of absorption) but fragmented the new repo into 31 mini-clusters vs the cold rebuild's ~8 (ARI 0.31) — the agglomerative threshold 0.3 was too strict. Two candidate fixes, judged by ARI against the cold-rebuild reference on the fmm chunks:
1. Agglomerative threshold sweep {0.3, 0.4, 0.5} — same deterministic mechanism, looser glue.
2. Scoped mini-pipeline: run the settled recipe (UMAP → HDBSCAN leaf → nearest-member assignment) on **just the novel pool** — deterministic for a fixed pool (fixed seed, no dependence on the existing tree), and by construction cannot touch existing clusters. UMAP's n_neighbors is clamped for the small pool.

In [22]:
tau = 0.70
absorbed22 = best_sim >= tau
novel22 = fmm_ids[~absorbed22]
ref_novel = np.array([int(labels_full_t[pos_full[g]]) for g in novel22])
print(f"novel pool: {len(novel22)} chunks; cold rebuild formed "
      f"{len(set(ref_novel))} clusters for them\n")

from sklearn.metrics import adjusted_rand_score

print("=== option 1: agglomerative threshold sweep ===")
for thr in [0.3, 0.4, 0.5]:
    agg = AgglomerativeClustering(n_clusters=None, distance_threshold=thr,
                                  metric="cosine", linkage="average").fit(combined_all[novel22])
    sizes22 = sorted(np.bincount(agg.labels_).tolist(), reverse=True)
    ari = adjusted_rand_score(ref_novel, agg.labels_)
    print(f"  thr={thr}: {agg.labels_.max() + 1} clusters (sizes {sizes22}), ARI vs cold: {ari:.3f}")

print("\n=== option 2: scoped mini-pipeline on the novel pool ===")
n_pool = len(novel22)
mini_reducer = umap.UMAP(n_components=min(UMAP_DIMS, max(2, n_pool - 2)),
                         n_neighbors=min(15, max(2, n_pool - 1)),
                         metric="cosine", random_state=42)
mini_coords = mini_reducer.fit_transform(combined_all[novel22])
mini_labels = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, metric="euclidean",
                      cluster_selection_method="leaf").fit_predict(mini_coords)
mini_noise = np.where(mini_labels == -1)[0]
mini_core = np.where(mini_labels >= 0)[0]
if len(mini_noise) and len(mini_core):
    sims_mini = combined_all[novel22[mini_noise]] @ combined_all[novel22[mini_core]].T
    mini_labels[mini_noise] = mini_labels[mini_core[sims_mini.argmax(axis=1)]]
sizes_mini = sorted(np.bincount(mini_labels).tolist(), reverse=True)
ari_mini = adjusted_rand_score(ref_novel, mini_labels)
print(f"  {mini_labels.max() + 1} clusters (sizes {sizes_mini}), ARI vs cold: {ari_mini:.3f}")

print("\n=== mini-pipeline clusters, named ===")
for l in range(mini_labels.max() + 1):
    members = novel22[mini_labels == l]
    segs = Counter("/".join(all_paths[g].split("/")[2:5]) for g in members)
    top, n = segs.most_common(1)[0]
    print(f"  NEW_{l} ({len(members)}): {top} [{n}/{len(members)}]")
    for g in sorted(members, key=lambda g: all_paths[g])[:4]:
        print(f"      {all_paths[g]}")
    if len(members) > 4:
        print(f"      ... +{len(members) - 4} more")

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


novel pool: 90 chunks; cold rebuild formed 6 clusters for them

=== option 1: agglomerative threshold sweep ===
  thr=0.3: 31 clusters (sizes [18, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1]), ARI vs cold: 0.314
  thr=0.4: 18 clusters (sizes [22, 14, 8, 7, 7, 5, 4, 4, 4, 3, 2, 2, 2, 2, 1, 1, 1, 1]), ARI vs cold: 0.545
  thr=0.5: 7 clusters (sizes [51, 24, 5, 4, 3, 2, 1]), ARI vs cold: 0.466

=== option 2: scoped mini-pipeline on the novel pool ===


  5 clusters (sizes [29, 17, 17, 15, 12]), ARI vs cold: 0.810

=== mini-pipeline clusters, named ===
  NEW_0 (15): src/main/java [3/15]
      branow/football-match-mvc/.gitattributes
      branow/football-match-mvc/.gitignore
      branow/football-match-mvc/README.md
      branow/football-match-mvc/build.gradle
      ... +11 more
  NEW_1 (17): src/main/java [17/17]
      branow/football-match-mvc/src/main/java/com/footballtournaments/config/AdminConfig.java
      branow/football-match-mvc/src/main/java/com/footballtournaments/controllers/AuthController.java
      branow/football-match-mvc/src/main/java/com/footballtournaments/controllers/HomeController.java
      branow/football-match-mvc/src/main/java/com/footballtournaments/controllers/MainController.java
      ... +13 more
  NEW_2 (29): src/main/java [29/29]
      branow/football-match-mvc/src/main/java/com/footballtournaments/controllers/DashboardController.java
      branow/football-match-mvc/src/main/java/com/footballtournaments/

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


Step 23: the order-bias proof. Claim under test: a tree grown *incrementally* (never re-running the pipeline over existing content) does not degrade versus a tree built cold from scratch on the same final data — and where it does deviate, the staleness monitor (misfit rate) detects it. Sequence: cold-build webphone alone → incrementally add football-match-mvc (assignment TAU 0.7 + scoped mini-pipeline, per step 22) → incrementally add the octocat chunks (pool of 4 < min_cluster_size, so forced nearest-assignment — the honest weak spot, noted). Compare the final incremental partition vs the cold full build (step 19):
- `ARI` over all 510 chunks — global agreement between grown and rebuilt
- `path_coherence` of each — absolute quality, not just agreement
- `misfit_rate` of each — the monitor: share of chunks sitting closer to another cluster's centroid than their own (pure dot products, deterministic, cheap enough to run continuously). The *gap* between incremental and cold misfit = measured staleness; the monitor passes if deviation from cold shows up here.

In [23]:
def misfit_rate(idx: np.ndarray, labels: np.ndarray) -> float:
    """Share of chunks more similar to another cluster's centroid than their own — the staleness monitor."""
    uniq = sorted(set(labels.tolist()))
    lab_pos = {l: k for k, l in enumerate(uniq)}
    cents = np.stack([combined_all[idx[labels == l]].mean(axis=0) for l in uniq])
    cents /= np.linalg.norm(cents, axis=1, keepdims=True)
    sims_c = combined_all[idx] @ cents.T
    own = sims_c[np.arange(len(idx)), [lab_pos[l] for l in labels]]
    sims_c[np.arange(len(idx)), [lab_pos[l] for l in labels]] = -np.inf
    return float((sims_c.max(axis=1) > own).mean())


def mini_pipeline(pool_ids: np.ndarray, offset: int) -> np.ndarray:
    red = umap.UMAP(n_components=min(UMAP_DIMS, max(2, len(pool_ids) - 2)),
                    n_neighbors=min(15, max(2, len(pool_ids) - 1)),
                    metric="cosine", random_state=42).fit_transform(combined_all[pool_ids])
    labs = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, metric="euclidean",
                   cluster_selection_method="leaf").fit_predict(red)
    noise_p, core_p = np.where(labs == -1)[0], np.where(labs >= 0)[0]
    if len(noise_p) and len(core_p):
        s = combined_all[pool_ids[noise_p]] @ combined_all[pool_ids[core_p]].T
        labs[noise_p] = labs[core_p[s.argmax(axis=1)]]
    return labs + offset


def incremental_add(idx: np.ndarray, labels: np.ndarray, new_ids: np.ndarray, tau=0.70):
    sims = combined_all[new_ids] @ combined_all[idx].T
    best, nearest = sims.max(axis=1), labels[sims.argmax(axis=1)]
    absorbed = best >= tau
    new_labels = np.where(absorbed, nearest, -1)
    novel = new_ids[~absorbed]
    if len(novel) >= MIN_CLUSTER_SIZE:
        new_labels[~absorbed] = mini_pipeline(novel, labels.max() + 1)
    elif len(novel):  # pool too small to form clusters: forced nearest-assignment (weak spot)
        new_labels[~absorbed] = nearest[~absorbed]
    return np.concatenate([idx, new_ids]), np.concatenate([labels, new_labels])


webphone_ids = np.array([i for i, p in enumerate(all_paths) if p.startswith("branow/webphone")])
fmm_all = np.array([i for i, p in enumerate(all_paths) if p.startswith("branow/football")])
octo_ids = np.array([i for i, p in enumerate(all_paths) if p.startswith("octocat/")])

# stage 0: cold base — webphone only
_, labels_wp = full_partition(webphone_ids)
print(f"stage 0 (webphone cold base): {len(set(labels_wp))} clusters, "
      f"misfit {misfit_rate(webphone_ids, labels_wp):.3f}")

# stage 1: + football-match-mvc, incrementally
idx_s1, labels_s1 = incremental_add(webphone_ids, labels_wp, fmm_all)
print(f"stage 1 (+fmm incremental):   {len(set(labels_s1))} clusters, "
      f"misfit {misfit_rate(idx_s1, labels_s1):.3f}")

# stage 2: + octocat chunks, incrementally
idx_s2, labels_s2 = incremental_add(idx_s1, labels_s1, octo_ids)
print(f"stage 2 (+octocat incremental): {len(set(labels_s2))} clusters, "
      f"misfit {misfit_rate(idx_s2, labels_s2):.3f}")

# cold reference on identical final data
order = np.argsort(idx_s2)
inc_final = labels_s2[order]           # labels aligned to idx 0..509
cold_final = labels_full_t             # step 19 cold total partition
from sklearn.metrics import adjusted_rand_score

prefixes_all = np.array(["/".join(p.split("/")[:3]) for p in all_paths])
def coherence_of(labels):
    agree = 0
    for l in set(labels.tolist()):
        c = Counter(prefixes_all[labels == l])
        agree += c.most_common(1)[0][1]
    return agree / len(labels)

print("\n=== grown-incrementally vs built-cold, same 510 chunks ===")
print(f"ARI: {adjusted_rand_score(inc_final, cold_final):.3f}")
print(f"clusters: incremental {len(set(inc_final.tolist()))} vs cold {len(set(cold_final.tolist()))}")
print(f"path_coherence: incremental {coherence_of(inc_final):.2f} vs cold {coherence_of(cold_final):.2f}")
print(f"misfit_rate (staleness monitor): incremental {misfit_rate(np.arange(len(all_paths)), inc_final):.3f} "
      f"vs cold {misfit_rate(np.arange(len(all_paths)), cold_final):.3f}")
print("verdict: monitor gap = staleness the incremental path accumulated; "
      "small gap + comparable coherence = order-bias not material at this scale")

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


stage 0 (webphone cold base): 31 clusters, misfit 0.039
stage 1 (+fmm incremental):   36 clusters, misfit 0.036
stage 2 (+octocat incremental): 36 clusters, misfit 0.035

=== grown-incrementally vs built-cold, same 510 chunks ===
ARI: 0.715
clusters: incremental 36 vs cold 37
path_coherence: incremental 0.95 vs cold 0.95
misfit_rate (staleness monitor): incremental 0.035 vs cold 0.031
verdict: monitor gap = staleness the incremental path accumulated; small gap + comparable coherence = order-bias not material at this scale


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
